# Recurrent Transformer Chess Player

## Architecture: Universal Transformer with Shared Weights & Iteration Embeddings

This notebook trains a **custom transformer from scratch** to predict chess moves.

### Neuroscience Motivation

The architecture is inspired by two findings from computational neuroscience:

1. **CORnet-s** (Kubilius et al., 2019): A 4-layer *recurrent* neural network that matches 100+ layer feedforward networks on ImageNet and achieves the highest Brain-Score. Recurrence allows shallow networks to match deep ones by processing the same representation multiple times.

2. **BLT Networks** (Spoerer et al., 2017): Bottom-up, Lateral, and Top-down connections improve performance on challenging visual recognition tasks. Lateral connections enable within-layer communication.

### Mapping to Transformers

| Neuroscience Concept | Transformer Equivalent |
|---|---|
| Lateral connections (BLT "L") | Self-attention within each layer |
| Recurrence (CORnet-s) | Shared transformer block applied K times |
| Time-cycle disambiguation | Learned iteration embeddings |
| Deep feedforward network | K iterations with 1 shared block ≈ K unique layers |

The model uses **classification** over all possible UCI moves (~4,272 classes), with **illegal move masking** at inference to guarantee legal output.

In [ ]:
# ── Setup (run once on Colab) ──────────────────────────────────────────────
# !pip install torch python-chess datasets huggingface_hub tqdm matplotlib

In [ ]:
import json
import os
import random
import time
from collections import Counter

import chess
import chess.pgn
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm

from model import (
    ChessTokenizer,
    RecurrentTransformer,
    build_move_vocabulary,
    MOVE_VOCAB,
    MOVE_TO_IDX,
    NUM_MOVES,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Move vocabulary size: {NUM_MOVES}")

## 1. Data Pipeline

We load chess games from the `angeluriot/chess_games` dataset, filter for games where both players are rated ≥ 1500 ELO, replay each game to extract `(FEN, UCI move)` pairs, and build a classification dataset.

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────
ds = load_dataset("angeluriot/chess_games", split="train", streaming=True)

In [ ]:
# ── Extract (FEN, UCI) pairs ───────────────────────────────────────────────

MIN_ELO = 1500
TARGET_POSITIONS = 750_000

positions = []  # list of (fen_str, uci_str)
games_processed = 0
games_skipped = 0

for row in tqdm(ds, desc="Processing games", total=None):
    if len(positions) >= TARGET_POSITIONS:
        break

    white_elo = row.get("white_elo")
    black_elo = row.get("black_elo")
    try:
        white_elo = int(white_elo) if white_elo is not None else 0
        black_elo = int(black_elo) if black_elo is not None else 0
    except (ValueError, TypeError):
        games_skipped += 1
        continue

    if white_elo < MIN_ELO or black_elo < MIN_ELO:
        games_skipped += 1
        continue

    # moves_uci is a list of UCI strings like ["e2e4", "e7e5", ...]
    uci_moves = row.get("moves_uci")
    if not uci_moves:
        games_skipped += 1
        continue

    board = chess.Board()
    for uci_str in uci_moves:
        fen = board.fen()
        try:
            move = chess.Move.from_uci(uci_str)
            if move not in board.legal_moves:
                break
        except (ValueError, chess.InvalidMoveError):
            break
        if uci_str in MOVE_TO_IDX:
            positions.append((fen, uci_str))
        board.push(move)

    games_processed += 1
    if games_processed % 5000 == 0:
        print(f"  {games_processed} games → {len(positions)} positions")

print(f"\nDone: {games_processed} games processed, {games_skipped} skipped")
print(f"Total positions: {len(positions)}")

In [ ]:
# ── Train / validation split ──────────────────────────────────────────────

random.seed(42)
random.shuffle(positions)

val_size = min(10_000, len(positions) // 20)
train_positions = positions[val_size:]
val_positions = positions[:val_size]

print(f"Train: {len(train_positions)}, Val: {len(val_positions)}")

## 2. Dataset & DataLoader

In [ ]:
tokenizer = ChessTokenizer(max_length=128)
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")


class ChessMoveDataset(Dataset):
    def __init__(self, data, tokenizer, move_to_idx):
        self.data = data
        self.tokenizer = tokenizer
        self.move_to_idx = move_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        fen, uci = self.data[idx]
        input_ids = torch.tensor(self.tokenizer.encode(fen), dtype=torch.long)
        label = self.move_to_idx[uci]
        return input_ids, label


train_ds = ChessMoveDataset(train_positions, tokenizer, MOVE_TO_IDX)
val_ds = ChessMoveDataset(val_positions, tokenizer, MOVE_TO_IDX)

BATCH_SIZE = 512

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 3. Model

```
FEN (char tokens) → [CLS] + Token Embedding + Positional Encoding
    → Shared Transformer Block × K iterations (with iteration embeddings)
    → [CLS] pooling → Linear → logits over 4,272 possible UCI moves
```

In [ ]:
model = RecurrentTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=256,
    nhead=8,
    d_ff=1024,
    num_iterations=6,
    num_moves=NUM_MOVES,
    max_seq_len=128,
    dropout=0.1,
    pad_token_id=tokenizer.pad_id,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.1f} MB (fp32)")

## 4. Training

In [ ]:
NUM_EPOCHS = 8
LR = 3e-4
WEIGHT_DECAY = 0.01

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = NUM_EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
criterion = nn.CrossEntropyLoss()

print(f"Total training steps: {total_steps}")

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_top1": [], "val_top5": []}


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct_top1 = 0
    correct_top5 = 0
    total = 0
    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            logits = model(input_ids)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)

            _, top5_preds = logits.topk(5, dim=1)
            correct_top1 += (top5_preds[:, 0] == labels).sum().item()
            correct_top5 += (top5_preds == labels.unsqueeze(1)).any(dim=1).sum().item()
            total += labels.size(0)

    return total_loss / total, correct_top1 / total, correct_top5 / total


best_val_top1 = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_samples = 0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for input_ids, labels in pbar:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item() * labels.size(0)
        epoch_samples += labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    train_loss = epoch_loss / epoch_samples
    val_loss, val_top1, val_top5 = evaluate(model, val_loader)
    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_top1"].append(val_top1)
    history["val_top5"].append(val_top5)

    print(
        f"  Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | "
        f"Top-1: {val_top1:.4f} | Top-5: {val_top5:.4f} | {elapsed:.0f}s"
    )

    if val_top1 > best_val_top1:
        best_val_top1 = val_top1
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  ↑ New best model saved (top-1: {val_top1:.4f})")

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train", marker="o")
axes[0].plot(history["val_loss"], label="Val", marker="o")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["val_top1"], label="Top-1", marker="o")
axes[1].plot(history["val_top5"], label="Top-5", marker="o")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## 6. Evaluation: Play Test Games

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load("best_model.pt", map_location=device, weights_only=True))
model.eval()


@torch.no_grad()
def predict_move(model, tokenizer, fen):
    """Predict the best legal move for a given FEN position."""
    board = chess.Board(fen)
    legal_moves = list(board.legal_moves)
    if not legal_moves:
        return None

    tokens = tokenizer.encode(fen)
    input_ids = torch.tensor([tokens], device=device)
    logits = model(input_ids).squeeze(0)

    legal_uci = {m.uci() for m in legal_moves}
    mask = torch.full_like(logits, float("-inf"))
    for uci in legal_uci:
        idx = MOVE_TO_IDX.get(uci)
        if idx is not None:
            mask[idx] = 0.0
    logits = logits + mask

    best_idx = logits.argmax().item()
    return MOVE_VOCAB[best_idx]


# Play a quick game against random moves
def play_test_game(model, tokenizer, max_moves=200):
    board = chess.Board()
    move_count = 0
    model_color = chess.WHITE

    while not board.is_game_over() and move_count < max_moves:
        if board.turn == model_color:
            uci = predict_move(model, tokenizer, board.fen())
            if uci is None:
                break
            board.push_uci(uci)
        else:
            legal = list(board.legal_moves)
            board.push(random.choice(legal))
        move_count += 1

    return board.result(), move_count


results = Counter()
num_test_games = 20
for i in range(num_test_games):
    result, moves = play_test_game(model, tokenizer)
    results[result] += 1
    if i < 5:
        print(f"  Game {i+1}: {result} in {moves} moves")

print(f"\nResults over {num_test_games} games vs Random:")
for r, c in sorted(results.items()):
    print(f"  {r}: {c} ({100*c/num_test_games:.0f}%)")

## 7. Push to HuggingFace

In [ ]:
from huggingface_hub import HfApi, login

# Authenticate (set HF_TOKEN env var or call login())
login()

REPO_ID = "Izzent/recurrent-transformer-chess"

# Save artifacts
config = model.get_config()
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

with open("tokenizer.json", "w") as f:
    json.dump(tokenizer.to_dict(), f, indent=2)

torch.save(model.state_dict(), "model.pt")

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)

for fname in ["config.json", "tokenizer.json", "model.pt"]:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=fname,
        repo_id=REPO_ID,
    )
    print(f"Uploaded {fname}")

print(f"\nModel uploaded to https://huggingface.co/{REPO_ID}")

## 8. Final Validation with chess_tournament

Run this after installing the chess_exam package to confirm the player works end-to-end.

In [ ]:
# !git clone https://github.com/bylinina/chess_exam.git
# %cd chess_exam
# !pip install -e .
# %cd ..

In [ ]:
# from chess_tournament import Game, RandomPlayer
# from player import TransformerPlayer
#
# tp = TransformerPlayer("RecurrentTransformer")
# rp = RandomPlayer("Random")
#
# results = []
# for i in range(10):
#     game = Game(tp, rp, max_half_moves=200)
#     outcome, scores, fallbacks = game.play()
#     results.append((outcome, scores, fallbacks))
#     print(f"Game {i+1}: {outcome} | Fallbacks: {fallbacks}")
#
# total_fallbacks = sum(r[2].get("RecurrentTransformer", 0) for r in results)
# print(f"\nTotal fallbacks across 10 games: {total_fallbacks}")